In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "PCOS_data_without_infertility.xlsx"

df = pd.read_excel(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20, 4)


,Instructions to be followed :,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,NaN,NaN,NaN
1,1.0,Kindly ensure that the datas are converted to...,NaN,NaN
2,2.0,Please fill up the entire data set of a patient,NaN,NaN
3,3.0,Manipulated datas if any need to be highlighte...,NaN,NaN
4,4.0,"For every Yes/No questions *** , Indicate Yes...",NaN,NaN


In [2]:
print("Columns:")
for number, column in enumerate(df.columns, start=1):
    print(f"{number}. {column}")

print("\nData types:")
df.info()

Columns:
1. Instructions to be followed :
2. Unnamed: 1
3. Unnamed: 2
4. Unnamed: 3

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Instructions to be followed :  12 non-null     float64
 1   Unnamed: 1                     12 non-null     object 
 2   Unnamed: 2                     0 non-null      float64
 3   Unnamed: 3                     8 non-null      object 
dtypes: float64(2), object(2)
memory usage: 772.0+ bytes


In [3]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("Missing values:")
display(missing_values)

print("Duplicate rows:", df.duplicated().sum())

if "Patient File No." in df.columns:
    print(
        "Duplicate patient IDs:",
        df["Patient File No."].duplicated().sum()
    )

Missing values:


Unnamed: 2                       20
Unnamed: 3                       12
Unnamed: 1                        8
Instructions to be followed :     8
dtype: int64

Duplicate rows: 0


In [4]:
excel_file = pd.ExcelFile(DATA_PATH)

print("Sheet names:", excel_file.sheet_names)

for sheet in excel_file.sheet_names:
    sheet_df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    print(sheet, sheet_df.shape)

Sheet names: ['Instructions', 'Full_new']
Instructions (20, 4)
Full_new (541, 45)


In [5]:
df = pd.read_excel(
    DATA_PATH,
    sheet_name="Full_new"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (541, 45)


,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Unnamed: 44
0,1,1,0,28,44.6,152.0,19.300000,15,78,22,...,1.0,0,110,80,3,3,18.0,18.0,8.5,NaN
1,2,2,0,36,65.0,161.5,24.921163,15,74,20,...,0.0,0,120,70,3,5,15.0,14.0,3.7,NaN
2,3,3,1,33,68.8,165.0,25.270891,11,72,18,...,1.0,0,120,80,13,15,18.0,20.0,10.0,NaN
3,4,4,0,37,65.0,148.0,29.674945,13,72,20,...,0.0,0,120,70,2,2,15.0,14.0,7.5,NaN
4,5,5,0,25,52.0,161.0,20.060954,11,72,18,...,0.0,0,120,80,3,4,16.0,14.0,7.0,NaN


In [6]:
empty_columns = [
    column for column in df.columns
    if df[column].isna().all()
]

print("Completely empty columns:", empty_columns)

Completely empty columns: []


In [8]:
df_clean = df.drop(columns=empty_columns).copy()

print("Original shape:", df.shape)
print("Working shape:", df_clean.shape)

Original shape: (541, 45)
Working shape: (541, 45)


In [9]:
print("Target counts:")
display(df_clean["PCOS (Y/N)"].value_counts(dropna=False))

print("Target percentages:")
display(
    df_clean["PCOS (Y/N)"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("Duplicate rows:", df_clean.duplicated().sum())
print(
    "Duplicate patient IDs:",
    df_clean["Patient File No."].duplicated().sum()
)

Target counts:


PCOS (Y/N)
0    364
1    177
Name: count, dtype: int64

Target percentages:


PCOS (Y/N)
0    67.28
1    32.72
Name: proportion, dtype: float64

Duplicate rows: 0
Duplicate patient IDs: 0


In [10]:
missing_report = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percent": (
        df_clean.isna().mean() * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["missing_count"] > 0
].sort_values("missing_count", ascending=False)

display(missing_report)

,missing_count,missing_percent
Unnamed: 44,539,99.63
Marraige Status (Yrs),1,0.18
Fast food (Y/N),1,0.18


In [11]:
display(
    df.loc[
        df["Unnamed: 44"].notna(),
        ["Patient File No.", "Unnamed: 44"]
    ]
)

,Patient File No.,Unnamed: 44
180,181,.
363,364,7


In [12]:
display(
    df.loc[
        df["Marraige Status (Yrs)"].isna()
        | df["Fast food (Y/N)"].isna(),
        [
            "Patient File No.",
            "Marraige Status (Yrs)",
            "Fast food (Y/N)",
            "PCOS (Y/N)"
        ]
    ]
)

,Patient File No.,Marraige Status (Yrs),Fast food (Y/N),PCOS (Y/N)
156,157,5.0,NaN,0
458,459,NaN,0.0,1


In [13]:
column_report = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "unique_values": df.nunique(dropna=True).values
})

display(column_report)

,column,data_type,unique_values
0,Sl. No,int64,541
1,Patient File No.,int64,541
2,PCOS (Y/N),int64,2
3,Age (yrs),int64,29
4,Weight (Kg),float64,117
5,Height(Cm),float64,50
6,BMI,float64,355
7,Blood Group,int64,8
8,Pulse rate(bpm),int64,11
9,RR (breaths/min),int64,8


In [16]:
clean_df = df.drop(columns=["Unnamed: 44"]).copy()

print("Original shape:", df.shape)
print("Working shape:", clean_df.shape)

Original shape: (541, 45)
Working shape: (541, 44)


In [18]:
for column in clean_df.columns:
    if "HCG" in column.upper() or "AMH" in column.upper():
        print(repr(column))

'  I   beta-HCG(mIU/mL)'
'II    beta-HCG(mIU/mL)'
'AMH(ng/mL)'


In [19]:
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Column names cleaned.")

Column names cleaned.


In [20]:
for column in clean_df.columns:
    if "HCG" in column.upper() or "AMH" in column.upper():
        print(repr(column))

'I beta-HCG(mIU/mL)'
'II beta-HCG(mIU/mL)'
'AMH(ng/mL)'


In [21]:
for column in ["II beta-HCG(mIU/mL)", "AMH(ng/mL)"]:
    converted = pd.to_numeric(clean_df[column], errors="coerce")

    invalid = clean_df.loc[
        converted.isna() & clean_df[column].notna(),
        ["Patient File No.", column]
    ]

    print(f"\nInvalid values in {column}:")
    display(invalid)


Invalid values in II beta-HCG(mIU/mL):


,Patient File No.,II beta-HCG(mIU/mL)
123,124,1.99.



Invalid values in AMH(ng/mL):


,Patient File No.,AMH(ng/mL)
305,306,a


In [22]:
for column in [
    "Cycle(R/I)",
    "BP _Systolic (mmHg)",
    "BP _Diastolic (mmHg)"
]:
    print(f"\n{column}:")
    display(clean_df[column].value_counts(dropna=False).sort_index())


Cycle(R/I):


Cycle(R/I)
2    390
4    150
5      1
Name: count, dtype: int64


BP _Systolic (mmHg):


BP _Systolic (mmHg)
12       1
100     13
110    264
120    253
130      8
140      2
Name: count, dtype: int64


BP _Diastolic (mmHg):


BP _Diastolic (mmHg)
8        1
60       1
70     159
80     379
100      1
Name: count, dtype: int64

In [23]:
columns_to_describe = [
    "Age (yrs)",
    "Pulse rate(bpm)",
    "Cycle length(days)",
    "FSH(mIU/mL)",
    "LH(mIU/mL)",
    "FSH/LH",
    "TSH (mIU/L)",
    "Vit D3 (ng/mL)"
]

display(
    clean_df[columns_to_describe]
    .describe()
    .T[["min", "mean", "50%", "max"]]
)

,min,mean,50%,max
Age (yrs),20.000000,31.430684,31.000000,48.000000
Pulse rate(bpm),13.000000,73.247689,72.000000,82.000000
Cycle length(days),0.000000,4.940850,5.000000,12.000000
FSH(mIU/mL),0.210000,14.601832,4.850000,5052.000000
LH(mIU/mL),0.020000,6.469919,2.300000,2018.000000
FSH/LH,0.002146,6.904831,2.169231,1372.826087
TSH (mIU/L),0.040000,2.981281,2.260000,65.000000
Vit D3 (ng/mL),0.000000,49.915874,25.900000,6014.660000


In [24]:
suspicious_mask = (
    (clean_df["Pulse rate(bpm)"] < 40)
    | (clean_df["Cycle length(days)"] < 1)
    | (clean_df["Cycle(R/I)"] == 5)
    | (clean_df["BP _Systolic (mmHg)"] < 70)
    | (clean_df["BP _Diastolic (mmHg)"] < 40)
    | (clean_df["FSH(mIU/mL)"] > 100)
    | (clean_df["LH(mIU/mL)"] > 100)
    | (clean_df["TSH (mIU/L)"] > 20)
    | (clean_df["Vit D3 (ng/mL)"] > 200)
)

display(clean_df.loc[suspicious_mask])

,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm)
37,38,38,0,26,72.0,157.0,29.210110,11,70,18,...,0,0.0,0,120,80,1,2,6.5,8.5,10.0
39,40,40,0,20,74.0,171.0,25.306932,13,74,16,...,0,0.0,0,110,70,6,12,13.0,12.0,0.0
161,162,162,0,38,70.0,170.0,24.221453,13,75,20,...,0,0.0,0,12,80,1,1,11.0,12.0,5.0
191,192,192,1,29,63.0,153.0,26.912726,17,74,18,...,1,1.0,0,120,70,14,16,16.0,17.0,9.0
195,196,196,1,35,60.0,153.4,25.497672,13,72,20,...,1,1.0,0,120,80,8,10,15.0,13.0,5.5
200,201,201,0,30,47.0,158.0,18.827111,15,73,18,...,1,1.0,1,120,8,2,1,11.0,14.0,6.0
223,224,224,0,30,62.0,169.0,21.707923,11,18,20,...,0,1.0,1,120,70,4,3,14.0,18.0,8.7
233,234,234,0,31,50.0,152.0,21.641274,15,72,18,...,1,1.0,1,120,80,7,5,17.0,15.0,9.0
278,279,279,1,45,50.0,154.0,21.082813,17,72,18,...,1,1.0,0,120,80,4,8,15.0,18.0,7.2
296,297,297,0,31,50.0,155.0,20.811655,15,13,18,...,1,0.0,0,110,70,8,5,17.0,15.0,8.5


In [25]:
target_counts = clean_df["PCOS (Y/N)"].value_counts().sort_index()
target_percentages = (
    clean_df["PCOS (Y/N)"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

display(pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
}))

,count,percentage
PCOS (Y/N),,
0,364,67.28
1,177,32.72


In [26]:
print(
    "IDs contain identical values:",
    clean_df["Sl. No"].equals(clean_df["Patient File No."])
)

print("Duplicate Sl. No:", clean_df["Sl. No"].duplicated().sum())
print(
    "Duplicate Patient File No.:",
    clean_df["Patient File No."].duplicated().sum()
)

IDs contain identical values: True
Duplicate Sl. No: 0
Duplicate Patient File No.: 0


In [27]:
calculated_bmi = clean_df["Weight (Kg)"] / (
    clean_df["Height(Cm)"] / 100
) ** 2

calculated_fsh_lh = (
    clean_df["FSH(mIU/mL)"] / clean_df["LH(mIU/mL)"]
)

calculated_waist_hip = (
    clean_df["Waist(inch)"] / clean_df["Hip(inch)"]
)

print("Largest BMI difference:", (clean_df["BMI"] - calculated_bmi).abs().max())
print("Largest FSH/LH difference:", (clean_df["FSH/LH"] - calculated_fsh_lh).abs().max())
print(
    "Largest Waist:Hip difference:",
    (clean_df["Waist:Hip Ratio"] - calculated_waist_hip).abs().max()
)

Largest BMI difference: 1.5749999999999957
Largest FSH/LH difference: 0.025000000000000355
Largest Waist:Hip difference: 0.0008648648648649226


In [28]:
checks = {
    "Very low pulse": ("Pulse rate(bpm)", clean_df["Pulse rate(bpm)"] < 40),
    "Cycle length below 1": ("Cycle length(days)", clean_df["Cycle length(days)"] < 1),
    "Unexpected cycle code": ("Cycle(R/I)", clean_df["Cycle(R/I)"] == 5),
    "Very low systolic BP": ("BP _Systolic (mmHg)", clean_df["BP _Systolic (mmHg)"] < 70),
    "Very low diastolic BP": ("BP _Diastolic (mmHg)", clean_df["BP _Diastolic (mmHg)"] < 40),
    "FSH above 100": ("FSH(mIU/mL)", clean_df["FSH(mIU/mL)"] > 100),
    "LH above 100": ("LH(mIU/mL)", clean_df["LH(mIU/mL)"] > 100),
    "TSH above 20": ("TSH (mIU/L)", clean_df["TSH (mIU/L)"] > 20),
    "Vitamin D3 above 200": ("Vit D3 (ng/mL)", clean_df["Vit D3 (ng/mL)"] > 200),
}

issues = []

for issue, (column, mask) in checks.items():
    for _, row in clean_df.loc[mask, ["Patient File No.", column]].iterrows():
        issues.append({
            "Patient File No.": row["Patient File No."],
            "issue": issue,
            "column": column,
            "value": row[column]
        })

issues_df = pd.DataFrame(issues)
display(issues_df.sort_values("Patient File No."))

,Patient File No.,issue,column,value
8,38.0,TSH above 20,TSH (mIU/L),65.00
2,40.0,Cycle length below 1,Cycle length(days),0.00
4,162.0,Very low systolic BP,BP _Systolic (mmHg),12.00
12,192.0,Vitamin D3 above 200,Vit D3 (ng/mL),6014.66
13,196.0,Vitamin D3 above 200,Vit D3 (ng/mL),5418.60
5,201.0,Very low diastolic BP,BP _Diastolic (mmHg),8.00
0,224.0,Very low pulse,Pulse rate(bpm),18.00
9,234.0,TSH above 20,TSH (mIU/L),25.91
10,279.0,TSH above 20,TSH (mIU/L),22.59
1,297.0,Very low pulse,Pulse rate(bpm),13.00


In [29]:
bmi_check = clean_df[
    ["Patient File No.", "Weight (Kg)", "Height(Cm)", "BMI"]
].copy()

bmi_check["Calculated BMI"] = calculated_bmi
bmi_check["Difference"] = (
    bmi_check["BMI"] - bmi_check["Calculated BMI"]
).abs()

display(bmi_check.sort_values("Difference", ascending=False).head(10))

,Patient File No.,Weight (Kg),Height(Cm),BMI,Calculated BMI,Difference
440,441,56.0,160.0,20.3,21.875000,1.575000
402,403,65.0,158.0,25.0,26.037494,1.037494
481,482,66.1,148.0,30.8,30.177137,0.622863
412,413,55.0,152.0,23.2,23.805402,0.605402
517,518,59.0,152.0,25.2,25.536704,0.336704
313,314,50.0,161.0,19.1,19.289379,0.189379
415,416,60.0,160.0,23.5,23.437500,0.062500
421,422,74.1,164.0,27.6,27.550565,0.049435
306,307,40.0,155.0,16.6,16.649324,0.049324
353,354,55.0,164.0,20.4,20.449137,0.049137


In [31]:
quality_summary = pd.DataFrame({
    "issue": [
        "Missing marriage status",
        "Missing fast-food value",
        "Invalid II beta-HCG",
        "Invalid AMH"
    ],
    "count": [1, 1, 1, 1]
})

display(quality_summary)

,issue,count
0,Missing marriage status,1
1,Missing fast-food value,1
2,Invalid II beta-HCG,1
3,Invalid AMH,1


## Initial audit findings

- Dataset contains 541 records and 45 original columns.
- `Unnamed: 44` is a spreadsheet artifact and should be removed.
- `Sl. No` and `Patient File No.` contain identical unique identifiers.
- Target distribution: 364 non-PCOS and 177 PCOS records.
- One value is missing from `Marraige Status (Yrs)`.
- One value is missing from `Fast food (Y/N)`.
- `II beta-HCG(mIU/mL)` contains the invalid value `1.99.`
- `AMH(ng/mL)` contains the invalid value `a`.
- Suspected data-entry errors exist in pulse, blood pressure, cycle encoding, FSH, LH and Vitamin D3.
- Some stored BMI values differ from BMI recalculated using weight and height.
- No values have been corrected or imputed during this audit.